In [46]:
import diff_diff
import pandas as pd
from diff_diff import CallawaySantAnna, SunAbraham, ImputationDiD
from diff_diff import compute_honest_did

In [45]:
print(diff_diff.get_llm_guide("practitioner"))

# diff-diff Practitioner Guide

> An 8-step workflow for rigorous Difference-in-Differences analysis, based on
> Baker et al. (2025) "Difference-in-Differences Designs: A Practitioner's
> Guide" and adapted for the diff-diff library. Some steps are reorganized or
> extended relative to the paper:
>
> - **Numbering**: diff-diff uses 1-Define, 2-Assumptions, 3-Test PT,
>   4-Choose estimator, 5-Estimate, 6-Sensitivity, 7-Heterogeneity,
>   8-Robustness. The paper uses 1-Define, 2-Assumptions, 3-Estimation method,
>   4-Uncertainty, 5-Estimate, 6-Sensitivity, 7-Heterogeneity, 8-Keep learning.
> - **Parallel trends testing** is a separate Step 3 (the paper embeds it in
>   Step 2), to ensure AI agents execute it as a distinct action.
> - **Sources of uncertainty** (paper's Step 4) are folded into Step 5
>   (Estimate) with an explicit cluster-count check directive: >= 50 clusters
>   for asymptotic SEs, otherwise wild bootstrap. The 50-cluster threshold is
>   a diff-diff convention.
> - *

In [39]:
# Keep only observations within 24 months before treatment for treated states
panel = pd.read_csv("../data/processed/panel.csv")
panel["date"]=pd.to_datetime(panel["date"])
panel["first_launch"]=pd.to_datetime(panel["first_launch"])


In [40]:
print(panel['State'].nunique())

46


## Target Parameters
**Estimand**: Average Treatment Effect on the Treated (ATT) - Average effect of sports betting legalization on gambling helpline contacts per 100k, among states that legalized.

**Weighting**: Unweighted (equal weight per state) because helpline volume is normalized by population.

**Heterogeneity**: Do states that legalized early show different effects than states that legalized late?

**Event Study**: Dynamic effects by month relative to launch

## Parallel Trends Event Study
Each treated cohort follows parallel trends with never-treated states conditional on pre-PASPA baseline helpline volume. Implemented via the doubly robust estimator (consistent if either the outcome model or propensity score is correctly specified).


In [3]:
cs = CallawaySantAnna(
    control_group='never_treated',
    estimation_method='dr',
    cluster='State',
    n_bootstrap=999,
)

In [4]:
results_es = cs.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
    aggregate='event_study'
)

/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=30. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=32. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(


In [5]:
if results_es.event_study_effects:
    for rel_t, eff in sorted(results_es.event_study_effects.items()):
        if rel_t < 0:
            print(f"Pre-period {rel_t}: ATT={eff['effect']:.4f}, SE={eff['se']:.4f}")

Pre-period -97: ATT=1.1281, SE=0.9647
Pre-period -96: ATT=0.6212, SE=0.2731
Pre-period -95: ATT=-1.9110, SE=1.7259
Pre-period -94: ATT=2.5808, SE=2.1619
Pre-period -93: ATT=-1.7208, SE=1.5273
Pre-period -92: ATT=0.1747, SE=0.2302
Pre-period -91: ATT=-0.8534, SE=1.2205
Pre-period -90: ATT=-0.4782, SE=0.5336
Pre-period -89: ATT=-1.5848, SE=1.0175
Pre-period -88: ATT=2.0746, SE=0.8486
Pre-period -87: ATT=-1.5604, SE=0.7864
Pre-period -86: ATT=-0.1658, SE=0.3308
Pre-period -85: ATT=0.6241, SE=0.8786
Pre-period -84: ATT=0.2390, SE=0.5453
Pre-period -83: ATT=-0.2140, SE=0.3738
Pre-period -82: ATT=-0.5908, SE=0.5452
Pre-period -81: ATT=0.2066, SE=0.1959
Pre-period -80: ATT=0.0675, SE=0.4277
Pre-period -79: ATT=-0.0088, SE=0.2493
Pre-period -78: ATT=-0.2401, SE=0.2389
Pre-period -77: ATT=0.9758, SE=0.3291
Pre-period -76: ATT=0.4394, SE=0.5784
Pre-period -75: ATT=-0.9338, SE=0.6049
Pre-period -74: ATT=-0.0498, SE=0.3272
Pre-period -73: ATT=0.2122, SE=0.4384
Pre-period -72: ATT=0.8721, SE=0.8138

In [6]:
for rel_t, eff in sorted(results_es.event_study_effects.items()):
    if rel_t >= 0:
        print(f"Post-period {rel_t:4d}: ATT={eff['effect']:.4f}, SE={eff['se']:.4f}")

Post-period    0: ATT=1.8613, SE=0.6135
Post-period    1: ATT=2.7744, SE=1.0349
Post-period    2: ATT=1.3050, SE=0.5363
Post-period    3: ATT=1.0400, SE=0.5058
Post-period    4: ATT=1.0763, SE=0.5313
Post-period    5: ATT=0.4985, SE=0.3784
Post-period    6: ATT=0.8271, SE=0.5374
Post-period    7: ATT=0.1684, SE=0.4320
Post-period    8: ATT=0.5208, SE=0.5429
Post-period    9: ATT=0.6380, SE=0.5735
Post-period   10: ATT=0.2394, SE=0.6135
Post-period   11: ATT=0.7210, SE=0.6769
Post-period   12: ATT=0.1876, SE=0.7167
Post-period   13: ATT=-0.1508, SE=0.6028
Post-period   14: ATT=0.0476, SE=0.5808
Post-period   15: ATT=-0.1633, SE=0.5450
Post-period   16: ATT=0.2095, SE=0.6328
Post-period   17: ATT=0.4175, SE=0.7257
Post-period   18: ATT=0.8588, SE=0.7175
Post-period   19: ATT=0.8278, SE=0.6434
Post-period   20: ATT=0.8403, SE=0.6806
Post-period   21: ATT=0.7748, SE=0.7535
Post-period   22: ATT=0.6551, SE=0.7689
Post-period   23: ATT=0.7080, SE=0.6626
Post-period   24: ATT=0.8119, SE=0.646

In [7]:
print(f"Overall ATT: {results_es.overall_att:.4f}")
print(f"Overall SE:  {results_es.overall_se:.4f}")
print(results_es.summary())

Overall ATT: 1.3723
Overall SE:  0.5220
            Callaway-Sant'Anna Staggered Difference-in-Differences Results           

Total observations:                  5520
Treated units:                         27
Never-treated units:                   19
Treatment cohorts:                     21
Time periods:                         120
Control group:                 never_treated
Base period:                      varying

-------------------------------------------------------------------------------------
                   Overall Average Treatment Effect on the Treated                   
-------------------------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|   Sig.
-------------------------------------------------------------------------------------
ATT                   1.3723       0.5220      2.629     0.0080     **
-------------------------------------------------------------------------------------

95

## Estimation

In [8]:
results = cs.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
    aggregate='all',
)
print(results.summary())

/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=30. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=32. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(


            Callaway-Sant'Anna Staggered Difference-in-Differences Results           

Total observations:                  5520
Treated units:                         27
Never-treated units:                   19
Treatment cohorts:                     21
Time periods:                         120
Control group:                 never_treated
Base period:                      varying

-------------------------------------------------------------------------------------
                   Overall Average Treatment Effect on the Treated                   
-------------------------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|   Sig.
-------------------------------------------------------------------------------------
ATT                   1.3723       0.5324      2.578     0.0060     **
-------------------------------------------------------------------------------------

95% Confidence Interval: [0.3632, 2.4152]


/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=99. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/var/folders/6p/5s7l8jtj05s_1p_1jxzg33480000gn/T/ipykernel_59139/1245163128.py:1: UserWarning: Low Events Per Variable (EPV) detected in propensity score estimation for 2499 of 2499 cell(s). Minimum EPV = 1.0 (cohort g=30). Consider estimation_method='reg' (avoids propensity scores) or reducing the number of covariates. See results.epv_summary() for details.
  results = cs.fit(


## HonestDiD Sensitivity

In [ ]:
honest_rm = compute_honest_did(
    results,
    method='relative_magnitude',
    M=1.0) 

print(honest_rm.summary())

               Honest DiD Sensitivity Analysis Results                
                       (Rambachan & Roth 2023)                        

Method:                        Relative Magnitudes (Delta^RM)
Target:                        Equal-weight avg over post horizons
Restriction parameter (M):     1.0000
CI method:                     FLCI

----------------------------------------------------------------------
              Original Estimate (under parallel trends)               
----------------------------------------------------------------------
Point estimate:                1.8649
Standard error:                0.1281

----------------------------------------------------------------------
               Robust Results (allowing for violations)               
----------------------------------------------------------------------
Identified set:                [-204.7597, 208.4896]
95% Robust CI:                 [-205.0177, 208.7476]

Effect robust to violations:   No

Pre hori

/var/folders/6p/5s7l8jtj05s_1p_1jxzg33480000gn/T/ipykernel_59139/4108896745.py:1: UserWarning: HonestDiD on bootstrap-fitted CallawaySantAnna results uses a diagonal covariance matrix (cross-event-time covariance is not available from bootstrap). For full covariance structure, use analytical SEs (n_bootstrap=0).
  honest_rm = compute_honest_did(


## Robustness

In [48]:
sa = SunAbraham()
sa_result = sa.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
)

bjs = ImputationDiD()
bjs_result = bjs.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
)

print(f"CS  ATT: {results.overall_att:.4f} (SE: {results.overall_se:.4f})")
print(f"SA  ATT: {sa_result.overall_att:.4f} (SE: {sa_result.overall_se:.4f})")
print(f"BJS ATT: {bjs_result.overall_att:.4f} (SE: {bjs_result.overall_se:.4f})")

/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/sun_abraham.py:907: UserWarning: Regressor(s) ['baseline_contacts_per_100k'] are collinear with the absorbed fixed effects (unit 'State' and time 'period' fixed effects): their within-transformed values are numerically zero (relative projection residual <= 1e-10), so their coefficients are not identified and will be reported as NaN.
  ) = self._fit_saturated_regression(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/linalg.py:3882: UserWarning: Rank-deficient design matrix: dropping 1 of 2500 columns (column 2499). Coefficients for these columns are set to NA. This may indicate multicollinearity in your model specification.
  coefficients, residuals, fitted, vcov = solve_ols(


CS  ATT: 1.3723 (SE: 0.5324)
SA  ATT: 1.1462 (SE: 0.5142)
BJS ATT: 1.6462 (SE: 0.4308)


In [49]:
cs_nyt = CallawaySantAnna(
    control_group='not_yet_treated',
    estimation_method='reg',
    cluster='State',
    n_bootstrap=999,
)

results_nyt = cs_nyt.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
    aggregate='all',
)

print(f"CS never_treated ATT:   {results.overall_att:.4f} (SE: {results.overall_se:.4f})")
print(f"CS not_yet_treated ATT: {results_nyt.overall_att:.4f} (SE: {results_nyt.overall_se:.4f})")

CS never_treated ATT:   1.3723 (SE: 0.5324)
CS not_yet_treated ATT: 1.2163 (SE: 0.5536)


In [50]:
panel_no_covid = panel[
    ~panel['date'].between('2020-01-01', '2020-12-01')
].copy()

results_no_covid = cs.fit(
    panel_no_covid,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
    aggregate='all',
)

print(f"CS full panel:    {results.overall_att:.4f} (SE: {results.overall_se:.4f})")
print(f"CS drop 2020:     {results_no_covid.overall_att:.4f} (SE: {results_no_covid.overall_se:.4f})")

/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=30. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=32. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(


CS full panel:    1.3723 (SE: 0.5324)
CS drop 2020:     1.4388 (SE: 0.5721)


/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=95. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Near-separation detected in propensity score model: 5 of 20 observations have predicted probabilities within 1e-05 of 0 or 1. ATT estimates may be sensitive to model specification.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3428: UserWarning: Propensity scores for 

In [53]:
es_df = pd.DataFrame([
    {'relative_period': k, 'effect': v['effect'], 'se': v['se']}
    for k, v in sorted(results.event_study_effects.items())
])

es_df.to_csv("../data/processed/event_study_results.csv", index=False)
